# `generate_simulated_data.ipynb`

Connor Shrader

This file is used to generate model simulations under a variety of different parameter regimes. The resulting data is saved to the `simulated_data/` directory.

### Import packages & define helper functions

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from tqdm import trange, tqdm
from itertools import product
import os

import testis_model as tm
from saving_loading import combine_and_save, scenario_exists

In [2]:
def plot_grid(data, scenario):
    fig, ax = plt.subplots(2, 8, figsize = (12, 2.5), dpi = 300)

    ax2 = ax.ravel()
    for i in range(min(len(data), 16)):
        tm.visualize.plot(data[i], ax = ax2[i])
        ax2[i].set_xlabel('')
        ax2[i].set_ylabel('')
        ax2[i].set_xticks([])
        ax2[i].set_yticks([])

    save_path = os.path.join('scenarios', scenario, f'{scenario}.png')
    plt.savefig(save_path)
    plt.close()

    return fig, ax

In [3]:
def sample_ic(n_stem, n_clones, rng = np.random.default_rng()):
    """
    Set initial condition of model.

    This function samples an initial state (s1, s2, ..., sk) where
    k is the number of clones `n_clones` and s1+s2+...+sk = `n_stem`.
    The sampling is done uniformly on this space using stars and bars.
    """
    samples = []

    # Choose `n_clones - 1` cut points. We start at index 1 and use `replace = False`
    # to ensure that each clone has at least one member.
    cuts = rng.choice(range(1, n_stem), size = n_clones - 1, replace = False)
    cuts.sort()

    clone_sizes = [cuts[0]]
    for i in range(1, n_clones - 1):
        clone_sizes.append(cuts[i] - cuts[i - 1])
    clone_sizes.append(n_stem - cuts[-1])

    return np.array(clone_sizes)

In [4]:
sample_times = np.array(list(tm.data_processing.sample_map_int.values()))
print(f'There are {len(sample_times)} sample times: {sample_times}.')

There are 18 sample times: [ 5  6  7  8  9 10 11 12 13 14 16 17 18 20 22 24 25 26].


## Sweep over values of $r$

In [5]:
r_event = 36
n_stem = 1000
r_diffs = [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 25, 30, 36]
r_decay = 6.28
tau = 0.2
sperm_per_diff = 631
n_clones = 10
n_sims = 100
n_sperm = 472

for r_diff in r_diffs:
    seed = hash((r_event, n_stem, r_diff, r_decay)) % (2**32)
    rng = np.random.default_rng(seed)
    
    p_diff = r_diff / r_event

    scenario = f'r{r_event}_rp{r_diff}'
    if scenario_exists(scenario, makedir = True):
        print(f'Skipping {scenario} - file already exists.')
        continue
    
    data = []
    for i in trange(n_sims, desc = f'{scenario}'):
        init_freq = sample_ic(n_stem, n_clones, rng = rng)
        
        hmm = tm.gillespie.AdvancedTestisModel(
            n_stem = n_stem, n_clones = n_clones, r_event = r_event, p_diff = p_diff,
            init_freq = init_freq, r_decay = r_decay, sperm_per_diff = sperm_per_diff,
            tau = tau
        )
    
        sample_times_fish = np.repeat(sample_times, 3)
        results, _, _ = hmm.simulate_gillespie(sample_times_fish, rng=rng, n_sperm = n_sperm)
        results.insert(0, 'scenario', scenario)
        results.insert(1, 'fish', i)
        results.insert(2, 'duplicate', 0)
        
        tm.data_processing.sort_results(results)
        data.append(results)

    df = combine_and_save(data, scenario)
    fig, ax = plot_grid(data, scenario);
    plt.close()

r36_rp36: 100%|█████████████████████████████████| 100/100 [01:07<00:00,  1.49it/s]


## Realistic sample & initial conditions

In [6]:
filepath = os.path.join('..', 'experiments', 'full_pipeline', 'final_data.csv')
all_data = pd.read_csv(filepath)

unique_fish = all_data['fish'].unique()
unique_fish = [f for f in unique_fish if 'ctrl' not in f]

fish_nums = sorted([int(f.replace('fish', '')) for f in unique_fish])
print(f'The unique fish found in the dataset are: {fish_nums}.')

The unique fish found in the dataset are: [4, 5, 10, 12, 13, 14, 15, 16, 17, 18, 22, 26, 35, 37, 38, 40, 42, 44, 45, 50, 51].


In [7]:
def fish_metadata(all_data, fish):
    """
    This function takes in a loaded dataset and a fish identifier. It returns the times at which the fish
    was sampled as well as the initial distribution of clones in that fish. These values are used to
    initialize simulated fish with the same sample times and initial distribution.
    """
    fish_data = all_data.query('fish == @fish')

    # Get sample times.
    sample_ids = np.sort(fish_data['sample'].unique())
    sample_times = [tm.data_processing.sample_map_int[sample_id] for sample_id in sample_ids]

    # Find initial distribution.
    min_sample = fish_data['sample'].min()
    init_reads = np.sort(fish_data.query('sample == @min_sample') \
                         .groupby('barcode', as_index = False) \
                         .agg({'reads': 'sum'})['reads'])
    
    init_counts = tm.data_processing.normalize_tuple(init_reads, 1000)

    return sample_times, init_counts

In [8]:
summ = 0
for f in fish_nums:
    _, init_counts = fish_metadata(all_data, f'fish{f}')
    # print(init_counts)
    summ += len(init_counts)

In [9]:
summ

180

In [10]:
r_events = [18, 36]
n_stem = 1000
r_diffs = [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 20, 25, 30, 36]
r_decay = 6.28
tau = 0.2
sperm_per_diff = 631
n_sims = 10
n_sperm = 472

for r_event, r_diff in product(r_events, r_diffs):
    seed = hash((r_event, n_stem, r_diff, r_decay)) % (2**32)
    rng = np.random.default_rng(seed)

    if r_event < r_diff: continue
    p_diff = r_diff / r_event

    scenario = f'r{r_event}_rp{r_diff}_mimic'
    if scenario_exists(scenario, makedir = True):
        print(f'Skipping {scenario} - file already exists.')
        continue

    data = []
    for fish in tqdm(fish_nums, desc = f'{scenario}'):
        sample_times, init_counts = fish_metadata(all_data, f'fish{fish}')
        sample_times_rep = np.repeat(sample_times, 3)
        n_clones = len(init_counts)
    
        for i in range(n_sims):
            init_freq = sample_ic(n_stem, n_clones, rng = rng)
            hmm = tm.gillespie.AdvancedTestisModel(
                n_stem = n_stem, n_clones = n_clones, r_event = r_event, p_diff = p_diff,
                init_freq = init_counts, r_decay = r_decay, sperm_per_diff = sperm_per_diff,
                tau = tau
            )

            results, _, _ = hmm.simulate_gillespie(sample_times_rep, rng=rng, n_sperm = n_sperm)
            results.insert(0, 'scenario', scenario)
            results.insert(1, 'fish', fish)
            results.insert(2, 'duplicate', i)
            tm.data_processing.sort_results(results)
            data.append(results)

    df = combine_and_save(data, scenario)
    fig, ax = plot_grid(data, scenario);
    plt.close()

r36_rp36_mimic: 100%|█████████████████████████████| 21/21 [01:40<00:00,  4.76s/it]


## Vary values of `n_stem`, `r_diff`, and `r_decay`. (currently unused)

In [11]:
r_event = 36
n_stems = [500, 1000, 1500]
r_diffs = [0, 1, 4, 15, 36]
r_decays = [3.14, 6.28, 9.42]
tau = 0.2
sperm_per_diff = 631
n_clones = 10
n_sims = 100
n_sperm = 472

for n_stem, r_diff, r_decay in product(n_stems, r_diffs, r_decays):
    seed = hash((n_stem, r_diff, r_decay)) % (2**32)
    rng = np.random.default_rng(seed)
    
    p_diff = r_diff / r_event

    scenario = f'n{n_stem}_r{r_diff}_y{r_decay:.0f}'
    if scenario_exists(scenario, makedir = True):
        print(f'Skipping {scenario} - file already exists.')
        continue
    
    data = []
    for i in trange(n_sims, desc = f'{scenario}'):
        init_freq = sample_ic(n_stem, n_clones, rng = rng)
        
        hmm = tm.gillespie.AdvancedTestisModel(
            n_stem = n_stem, n_clones = n_clones, r_event = r_event, p_diff = p_diff,
            init_freq = init_freq, r_decay = r_decay, sperm_per_diff = sperm_per_diff,
            tau = tau
        )
    
        sample_times_fish = np.repeat(sample_times, 3)
        results, _, _ = hmm.simulate_gillespie(sample_times_fish, rng=rng, n_sperm = n_sperm)
        results.insert(0, 'scenario', scenario)
        results.insert(1, 'fish', 0)
        results.insert(2, 'duplicate', i)
        
        tm.data_processing.sort_results(results)
        data.append(results)

    df = combine_and_save(data, scenario)
    fig, ax = plot_grid(data, scenario);
    plt.close()

n1500_r36_y9: 100%|█████████████████████████████| 100/100 [01:17<00:00,  1.29it/s]
